In [46]:
%load_ext autoreload
%autoreload 2 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [47]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import kaleido

### Experimental protocol:

We already have 4 datasets. We denote the four datasets as: OurAirports (A), Sakila (B), AdventureWorks (C), Oracle HR (D). Two scenarios are compared.

- Generic model: trained on 3 datasets, evaluated on the 4th one never seen before (ABC → D, BCD → A, ABD → C, ACD → B).
- Specialized model (baseline): trained and evaluated on the same dataset.

We have already set up the experiments to evaluate these cases. Now, we want to build a “matrix”:

|     | A | B | C | D |
| --- | - | - | - | - |
| ABC |   |   |   |   |
| ABD |   |   |   |   |
| ACD |   |   |   |   |
| CBD |   |   |   |   |
		

This is to observe performance when the training set remains fixed, but the test set changes. We want 3 matrices: AUROC, AUPRC, and F1.

In [48]:
base_results_path = "/home/gquetel/experiences-results/2026-02-12-results/output"

figures_dir = Path("../output/experiments/TL-matrix")
figures_dir.mkdir(parents=True, exist_ok=True)

In [49]:
def load_matrix_results(results_path: str, model_type: str, train_sets : list, test_sets : list) -> dict:
    """Loads the results for a model and returns 3 DataFrames (AUROC, AUPRC, F1)."""
    # Initialize the matrices
    auroc_matrix = pd.DataFrame(index=train_sets, columns=test_sets, dtype=float)
    auprc_matrix = pd.DataFrame(index=train_sets, columns=test_sets, dtype=float)
    f1_matrix = pd.DataFrame(index=train_sets, columns=test_sets, dtype=float)

    for train in train_sets:
        for test in test_sets:
            csv_path = Path(results_path) / f"{model_type}_{train}_on_{test}" / "results.csv"
            if csv_path.exists():
                df = pd.read_csv(csv_path)
                auroc_matrix.loc[train, test] = df["rocauc"].iloc[0]
                auprc_matrix.loc[train, test] = df["auprc"].iloc[0]
                # Parse F1 score (format "88.53%")
                f1_str = df["fone"].iloc[0]
                f1_matrix.loc[train, test] = float(f1_str.rstrip('%')) / 100

    return {"auroc": auroc_matrix, "auprc": auprc_matrix, "f1": f1_matrix}


In [50]:
def create_heatmap(matrix: pd.DataFrame, title: str, colorscale: str = "RdYlGn") -> go.Figure:
    """Creates a Plotly heatmap with annotations."""
    annotations = []
    for i, train in enumerate(matrix.index):
        for j, test in enumerate(matrix.columns):
            val = matrix.loc[train, test]
            is_generalization = test not in train
            annotations.append(dict(
                x=test, y=train,
                text=f"{val:.3f}" if not pd.isna(val) else "N/A",
                font=dict(
                    color="white" if val < 0.7 else "black",
                    size=14,
                    weight="bold" if is_generalization else "normal"
                ),
                showarrow=False
            ))

    fig = go.Figure(data=go.Heatmap(
        z=matrix.values,
        x=matrix.columns.tolist(),
        y=matrix.index.tolist(),
        colorscale=colorscale,
        zmin=0.5,
        zmax=1.0,
        showscale=True
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Test Dataset",
        yaxis_title="Training Set",
        annotations=annotations,
        width=500,
        height=400
    )
    return fig


def save_figure(fig: go.Figure, name: str) -> None:
    """Saves a Plotly figure as PDF, PNG, and interactive HTML."""
    # fig.write_image(figures_dir / f"{name}.pdf")
    fig.write_image(figures_dir / f"{name}.png", scale=2)

## AE-SecureBERT Results (Generic)

In [51]:
train_sets = ["ABC", "ABD", "ACD", "BCD"]
test_sets = ["A", "B", "C", "D"]

results_path = f"{base_results_path}/sbert_generic"
ae_sbert_results = load_matrix_results(results_path, "ae_sbert", train_sets, test_sets)

fig_auroc = create_heatmap(ae_sbert_results["auroc"], "AE-SecureBERT: AUROC Matrix")
fig_auprc = create_heatmap(ae_sbert_results["auprc"], "AE-SecureBERT: AUPRC Matrix")
fig_f1 = create_heatmap(ae_sbert_results["f1"], "AE-SecureBERT: F1 Matrix")

fig_auroc.show()
fig_auprc.show()
fig_f1.show()

save_figure(fig_auroc, "ae_sbert_auroc_generic")
save_figure(fig_auprc, "ae_sbert_auprc_generic")


## AE-SecureBERT Results (Specialised)

In [52]:
train_sets = ["D", "C", "B", "A"]
test_sets = ["A", "B", "C", "D"]

results_path = f"{base_results_path}/sbert_specialised"
ae_sbert_results = load_matrix_results(results_path, "ae_sbert", train_sets, test_sets)

fig_auroc = create_heatmap(ae_sbert_results["auroc"], "AE-SecureBERT: AUROC Matrix")
fig_auprc = create_heatmap(ae_sbert_results["auprc"], "AE-SecureBERT: AUPRC Matrix")
fig_f1 = create_heatmap(ae_sbert_results["f1"], "AE-SecureBERT: F1 Matrix")

fig_auroc.show()
fig_auprc.show()
fig_f1.show()

save_figure(fig_auroc, "ae_sbert_auroc_specialised")
save_figure(fig_auprc, "ae_sbert_auprc_specialised")

## AE-LI Results (Generic)

In [53]:
train_sets = ["ABC", "ABD", "ACD", "BCD"]
test_sets = ["A", "B", "C", "D"]

results_path = f"{base_results_path}/li_generic"
ae_li_results = load_matrix_results(results_path, "ae_li",train_sets,test_sets)

fig_auroc_li = create_heatmap(ae_li_results["auroc"], "AE-LI: AUROC Matrix")
fig_auprc_li = create_heatmap(ae_li_results["auprc"], "AE-LI: AUPRC Matrix")
fig_f1_li = create_heatmap(ae_li_results["f1"], "AE-LI: F1 Matrix")

fig_auroc_li.show()
fig_auprc_li.show()
fig_f1_li.show()

save_figure(fig_auroc_li, "ae_li_auroc_generic")
save_figure(fig_auprc_li, "ae_li_auprc_generic")


## AE-LI Results (Specialised)

In [54]:
train_sets = ["D", "C", "B", "A"]
test_sets = ["A", "B", "C", "D"]

results_path = f"{base_results_path}/li_specialised"
ae_li_results = load_matrix_results(results_path, "ae_li",train_sets,test_sets)

fig_auroc_li = create_heatmap(ae_li_results["auroc"], "AE-LI: AUROC Matrix")
fig_auprc_li = create_heatmap(ae_li_results["auprc"], "AE-LI: AUPRC Matrix")
fig_f1_li = create_heatmap(ae_li_results["f1"], "AE-LI: F1 Matrix")

fig_auroc_li.show()
fig_auprc_li.show()
fig_f1_li.show()

save_figure(fig_auroc_li, "ae_li_auroc_specialised")
save_figure(fig_auprc_li, "ae_li_auprc_specialised")